 # Análise e formatação de dados de vendas do PostgreSQL

 ## Objetivo

 Análise exploratória de dados de vendas: formatação e visualização estratégica para geração de insights e suporte à tomada de decisão.

 - Quais são os melhores 5 produtos de cada categoria com base no total de venda?
 - Como realizar a imputação de valores ausentes na coluna quantity através da modelagem do preço unitário por produto, região e descontos?


## Dicionário de dados:

![Print 1](imagens/SuperStore_ERD.png)

## Desenvolvimento

A tabela abaixo detalha a representatividade dos 5 principais produtos de cada categoria. Observa-se que o segmento de Tecnologia lidera o volume de vendas, enquanto o de Material de Escritório apresenta o menor desempenho no período analisado.

In [ ]:
SELECT *
FROM(
     SELECT pro.category,
	        pro.product_name,
	        ROUND(CAST(SUM(ord.sales) AS NUMERIC), 2) AS product_total_sales,
	        ROUND(CAST(SUM(ord.profit) AS NUMERIC), 2) AS product_total_profit,
	        RANK() OVER(PARTITION BY pro.category
	             ORDER BY SUM(ord.sales) DESC)
	             AS product_rank
     FROM orders AS ord
     INNER JOIN products AS pro
     ON pro.product_id = ord.product_id
	 GROUP BY pro.category, pro.product_name
     ORDER BY pro.product_name) AS ranked_results
WHERE product_rank <= 5
ORDER BY category ASC, product_total_sales DESC;

,category,product_name,product_total_sales,product_total_profit,product_rank
0,Furniture,"Hon Executive Leather Armchair, Adjustable",58193.48,5997.25,1
1,Furniture,"Office Star Executive Leather Armchair, Adjust...",51449.80,4925.80,2
2,Furniture,"Harbour Creations Executive Leather Armchair, ...",50121.52,10427.33,3
3,Furniture,"SAFCO Executive Leather Armchair, Black",41923.53,7154.28,4
4,Furniture,"Novimex Executive Leather Armchair, Adjustable",40585.13,5562.35,5
5,Office Supplies,"Eldon File Cart, Single Width",39873.23,5571.26,1
6,Office Supplies,"Hoover Stove, White",32842.60,-2180.63,2
7,Office Supplies,"Hoover Stove, Red",32644.13,11651.68,3
8,Office Supplies,"Rogers File Cart, Single Width",29558.82,2368.82,4
9,Office Supplies,"Smead Lockers, Industrial",28991.66,3630.44,5


O tratamento dos dados nulos foca na estimativa de volumetria, utilizando fatores de precificação relevantes (descontos, mercado e região) para inferir as quantidades faltantes com maior precisão.

In [ ]:
WITH missing AS (
	SELECT product_id,
		   discount, 
		   market,
		   region,
		   sales,
		   quantity
	FROM orders 
	WHERE quantity IS NULL
), 

unit_prices AS (SELECT o.product_id,
	                   CAST(o.sales / o.quantity AS NUMERIC) AS unit_price
FROM orders o
RIGHT JOIN missing AS m 
	ON o.product_id = m.product_id
	AND o.discount = m.discount
WHERE o.quantity IS NOT NULL
)

SELECT DISTINCT m.*,
	ROUND(CAST(m.sales AS NUMERIC) / up.unit_price,0) AS calculated_quantity
FROM missing AS m
INNER JOIN unit_prices AS up
	ON m.product_id = up.product_id;



,product_id,discount,market,region,sales,quantity,calculated_quantity
0,FUR-ADV-10000571,0.00,EMEA,EMEA,438.960,NaN,4
1,FUR-ADV-10004395,0.00,EMEA,EMEA,84.120,NaN,2
2,FUR-BO-10001337,0.15,US,West,308.499,NaN,3
3,TEC-STA-10003330,0.00,Africa,Africa,506.640,NaN,2
4,TEC-STA-10004542,0.00,Africa,Africa,160.320,NaN,4


## Conclusão

Portanto, a estruturação e o tratamento da consulta foram essenciais para garantir a integridade da base de dados, viabilizando análises mais consistentes e a geração de padrões, tendências ou comportamento relevantes sobre o comportamento de vendas.